# WoodelfHD, m=100 background: WoodelfHD, OriginalWoodelf and SHAP (notebook 1 of 2)

Runs `benchmarks/woodelfhd_m100_experiment.py` for every method except PLTreeSHAP.
PLTreeSHAP needs Python 3.8, so it runs in `woodelfhd_m100_2_pltreeshap.ipynb`, in a second Colab session at the same time.

**Runtime:** CPU with High-RAM. Estimated compute is about 9 h (WoodelfHD ~3 h, OriginalWoodelf ~1.4 h, SHAP ~3.8 h, model training ~0.5 h).
To split it across sessions, change `METHODS` in step 2.

### If the runtime crashes or disconnects
Reconnect and **run all cells from the top**. Nothing that finished is redone:
- **Results** are saved to Drive after every finished cell (`results/{method}.json`). Step 4 restores them and the run skips those cells.
- **Datasets** are prepared once (step 5) and kept on Drive, so HIGGS is not downloaded again.
- **Models** are trained straight into a Drive folder (`models_main/`), so they are not retrained.

Only the cell that was running when the runtime died is lost.

### Drive layout
```
MyDrive/treebranchmarks/woodelfhd_m100/
  results/       woodelf_hd.json, original_woodelf.json, shap.json
                 (+ pltreeshap_fasttreeshap.json from notebook 2)
  datasets/      prepared datasets, also used by notebook 2 (~1.1 GB)
  models_main/   this notebook's model cache
  woodelfhd_m100_experiment.html   the report (step 9)
```

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configuration ───────────────────────────────────────────────────
import os, pathlib

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_m100')
METHODS    = ['woodelf_hd', 'original_woodelf', 'shap']   # remove some to split the run across sessions

REPO_URL    = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'
REPO_BRANCH = 'main'
REPO        = pathlib.Path('/content/treebranchmarks')
TOOLS       = REPO / 'colab' / 'woodelfhd_m100_tools.py'   # restore / save-datasets / progress / report

RESULTS_DRIVE = DRIVE_ROOT / 'results'
RESULTS_DRIVE.mkdir(parents=True, exist_ok=True)
METHODS_STR = ' '.join(METHODS)
METHOD_ARGS = ' '.join(f'--method {m}' for m in METHODS)
print('Drive folder:', DRIVE_ROOT)
print('Methods     :', METHODS_STR)

In [ ]:
# ── Step 3: Clone the repo and install ──────────────────────────────────────
# woodelf_explainer is installed from PyPI as a dependency of treebranchmarks.
# To use the GitHub version instead, uncomment the last line.

if not REPO.exists():
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO}
if not (REPO / 'benchmarks' / 'woodelfhd_m100_experiment.py').exists() or not TOOLS.exists():
    raise FileNotFoundError(
        'The m=100 experiment files are not in the cloned repo. Push them to GitHub '
        '(or set REPO_BRANCH), delete /content/treebranchmarks and rerun this cell.')

!pip install -q -e {REPO}
# !pip install -q git+https://github.com/ron-wettenstein/woodelf.git

In [ ]:
# ── Step 4: Restore datasets, models and finished results from Drive ────────
!python {TOOLS} restore --drive-root "{DRIVE_ROOT}" --methods {METHODS_STR} --models-dir models_main

In [ ]:
# ── Step 5: Prepare datasets and keep them on Drive ─────────────────────────
# First run: downloads the four datasets (HIGGS takes ~20 min) and copies them to Drive.
# Later runs: nothing to do.
# Start notebook 2 once this step has finished, so it can copy the datasets instead of downloading them.
!python {TOOLS} save-datasets --drive-root "{DRIVE_ROOT}"

In [ ]:
# ── Step 6: Progress so far ─────────────────────────────────────────────────
!python {TOOLS} progress --drive-root "{DRIVE_ROOT}" --methods {METHODS_STR}

## Run

In [ ]:
# ── Step 7: Run the experiment ──────────────────────────────────────────────
# --result_location is a folder: each method rewrites its own {method}.json there after every finished cell.
os.chdir(REPO)
!python -u -m benchmarks.woodelfhd_m100_experiment {METHOD_ARGS} --result_location "{RESULTS_DRIVE}"

In [ ]:
# ── Step 8: Progress after the run ──────────────────────────────────────────
!python {TOOLS} progress --drive-root "{DRIVE_ROOT}" --methods {METHODS_STR}

## Report (after both notebooks have finished)

Builds the HTML report from all four result files on Drive. Nothing is run, trained or downloaded.
If any cell of any method is unfinished, it stops and lists those cells instead.

On a fresh runtime, run steps 1–3 first, then the cells below.

In [ ]:
# ── Step 9: Build the report ────────────────────────────────────────────────
!python {TOOLS} report --drive-root "{DRIVE_ROOT}"

In [ ]:
# ── Step 10: Download the report ────────────────────────────────────────────
from google.colab import files
files.download(str(DRIVE_ROOT / 'woodelfhd_m100_experiment.html'))